### Mounting Drive:

In [ ]:


from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📈 Metric-to-Price Change Correlation Analysis

This notebook performs a **quantitative comparison** between financial metrics and stock price movements across two consecutive quarters for companies in the Israeli stock market. It calculates the **alignment ("score")** between changes in financial indicators and the corresponding change in market price over defined time windows.

---

### 🎯 Goal

Identify how **well changes in financial metrics reflect stock price movements** over multiple time intervals after the release of following quarters earnings reports.

---

### 📂 Inputs

- **Quarterly Report CSVs**:
  - `report_analysis_2024_q3.csv`
  - `report_analysis_2024_q4.csv`
- **Ticker Mapping**: Maps Hebrew/English company names to TASE tickers.
- **Tickers List**: The list of TASE tickers to evaluate.
- **YFinance**: Used to retrieve historical closing prices.

---

### 🛠️ Processing Steps

1. **Load and Align**:
   - Extract companies with valid data in both quarters.
   - Convert company names to ticker symbols.
   - Match with `.TA` suffix in TASE tickers list.

2. **Determine Dates**:
   - Parse Q3 and Q4 report dates from filenames.
   - Used for anchoring price comparisons.

3. **Fetch Prices**:
   - For each ticker:
     - Retrieve price at Q3 and Q4 report dates (`get_closing_price`)
     - Calculate forward price movement over:
       - 1 day
       - 30 days
       - 90 days

4. **Compute Normalized Change**:
   - `normalize_change(val1, val2)` computes % change scaled to the base value.
   - Metrics and prices are normalized this way.

5. **Compare Changes**:
   - For each numeric metric in the CSV:
     - Compute Q3→Q4 metric % change
     - Compare to average price % change from both quarters
     - Compute **correlation score**:
       \[
       \text{Score} = 1 - \left| \text{Metric Change} - \text{Price Change} \right|
       \]
     - Clip score to [-1, 1], store "Positive"/"Negative" direction.

6. **Output**:
   - Aggregates all results into a single DataFrame.
   - Saves to:  
     `/Reports Parameters/Correlations/all_companies_correlation_summary.csv`

---

### 📈 Example Output (Per Row)

| Metric         | Interval          | Price Change % | Metric Change % | Score  | Direction | Company |
|----------------|-------------------|----------------|------------------|--------|-----------|---------|
| Revenue        | 1 month (30 days) | 4.23           | 6.02             | 0.9618 | Positive  | TEVA.TA |
| Operating Cash Flow | 90 days     | -2.12          | -1.98            | 0.9784 | Positive  | BZQ.TA  |

---

### ⚠️ Notes

- Skips companies without valid tickers or missing financial/price data.
- Automatically retries historical prices up to 5 days back to avoid weekends/holidays.
- Adds a 25s delay to avoid rate limits from YFinance.
- Correlation Score ≈ 1: strong alignment between metric and price.
- Score < 0: inverse movement (e.g., revenue ↑, price ↓).

---

### ✅ Result

- Final CSV contains a detailed breakdown of metric-price alignment across all valid companies and time intervals.
- Use this data to:
  - Identify financially reactive stocks
  - Build feature sets for predictive models
  - Generate insights on metric sensitivity



In [ ]:


import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
import time
# === CONFIGURATION ===
csv_path_1 = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/report_analysis_2024_q3.csv"
csv_path_2 = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/report_analysis_2024_q4.csv"
map_path = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Noam's work/Best model per stock/Helpful files /company_name_to_ticker.xlsx"

# === Load mapping: CompanyName → Ticker ===
map_df = pd.read_excel(map_path)
map_df.columns = map_df.columns.str.strip()
company_to_ticker = dict(zip(map_df['CompanyName'].str.upper(), map_df['Ticker'].str.upper()))
ticker_to_company = {v: k for k, v in company_to_ticker.items()}

# === Load full report CSVs to extract valid companies ===
full_df1 = pd.read_csv(csv_path_1)
full_df2 = pd.read_csv(csv_path_2)
full_df1.columns = full_df1.columns.str.strip()
full_df2.columns = full_df2.columns.str.strip()

# === Extract shared companies that exist in both quarters ===
report_companies_1 = full_df1['Company Name'].dropna().str.upper().unique()
report_companies_2 = full_df2['Company Name'].dropna().str.upper().unique()
common_companies = set(report_companies_1).intersection(report_companies_2)

# === Convert companies to tickers (from mapping) ===
valid_tickers_from_reports = {
    company_to_ticker[comp]
    for comp in common_companies
    if comp in company_to_ticker
}

# === Load tickers from list and filter by report-valid ones ===
tickers_df = pd.read_csv("/content/drive/Shareddrives/capstone project-stock market robo-advisor/tickers.csv")
tickers_list = tickers_df.iloc[:, 0].dropna().astype(str).str.upper().unique().tolist()

# Strip ".TA" to match the mapping
filtered_tickers = [
    t for t in tickers_list
    if t.replace(".TA", "") in valid_tickers_from_reports
]
tickers_list = filtered_tickers

# === Extract dates from filenames ===
def extract_date_from_filename(path):
    import re
    match = re.search(r'report_analysis_(\d{4})_q([1-4])', path.lower())
    if not match:
        raise ValueError("Filename format invalid. Expected: report_analysis_YYYY_qX.csv")
    year, quarter = int(match.group(1)), int(match.group(2))
    quarter_end_month = {1: "03-31", 2: "06-30", 3: "09-30", 4: "12-31"}
    return f"{year}-{quarter_end_month[quarter]}"

date1 = extract_date_from_filename(csv_path_1)
date2 = extract_date_from_filename(csv_path_2)

# === Utility functions ===
def normalize_change(val1, val2):
    if val1 == 0 or pd.isna(val1) or pd.isna(val2):
        return None
    return (val2 - val1) / abs(val1)

def get_closing_price(symbol, date_str):
    date = datetime.strptime(date_str, '%Y-%m-%d').date()
    for delta in range(5):  # Try up to 5 days back
        attempt_date = date - pd.Timedelta(days=delta)
        try:
            data = yf.download(symbol, start=attempt_date, end=attempt_date + pd.Timedelta(days=1),
                               interval="1d", progress=False, auto_adjust=True)
            if not data.empty:
                return data['Close'].iloc[0].item()
        except Exception as e:
            print(f"⚠️ Error for {symbol} on {attempt_date}: {e}")
        time.sleep(25)  # Avoid rate limiting
    return None

def get_price_change(symbol, base_date_str, days_offset):
    base_date = datetime.strptime(base_date_str, '%Y-%m-%d').date()
    end_date = base_date + pd.Timedelta(days=days_offset)
    data = yf.download(symbol, start=base_date, end=end_date + pd.Timedelta(days=1), interval="1d", progress=False, auto_adjust=True)
    if data.empty:
        return None
    return data['Close'].iloc[-1].item() - data['Close'].iloc[0].item()

# === Define intervals ===
intervals = {
    '1 Day': 1,
    '1 month (30 days)': 30,
    '1 quarter (90 days)': 90
}

# === Main processing loop ===
all_results = []

for ticker_symbol in tickers_list:  # You can remove the slice later
    print(f"🔍 Processing {ticker_symbol}...")
    try:
        base_ticker = ticker_symbol.replace(".TA", "")
        company_name = ticker_to_company.get(base_ticker, None)

        if not company_name:
            print(f"⚠️ Skipping {ticker_symbol} — no matching company.")
            continue

        row1 = full_df1[full_df1['Company Name'].str.upper() == company_name]
        row2 = full_df2[full_df2['Company Name'].str.upper() == company_name]

        if row1.empty or row2.empty:
            print(f"⚠️ Skipping {ticker_symbol} — missing company data in CSVs.")
            continue

        row1 = row1.select_dtypes(include=[np.number]).squeeze()
        row2 = row2.select_dtypes(include=[np.number]).squeeze()

        price1 = get_closing_price(ticker_symbol, date1)
        price2 = get_closing_price(ticker_symbol, date2)

        if price1 is None or price2 is None:
            print(f"⚠️ Skipping {ticker_symbol} — missing price data.")
            continue

        for label, days in intervals.items():
            price1_end = get_price_change(ticker_symbol, date1, days)
            price2_end = get_price_change(ticker_symbol, date2, days)

            if price1_end is None or price2_end is None:
                continue

            price_change_1 = normalize_change(price1, price1 + price1_end)
            price_change_2 = normalize_change(price2, price2 + price2_end)

            if price_change_1 is None or price_change_2 is None:
                continue

            avg_price_change = np.mean([price_change_1, price_change_2])

            for metric in row1.index:
                val1 = row1[metric]
                val2 = row2.get(metric, np.nan)
                metric_change = normalize_change(val1, val2)

                if metric_change is None:
                    continue

                similarity = 1 - abs(metric_change - avg_price_change)
                similarity = np.clip(similarity, -1, 1)

                all_results.append({
                    "Metric": metric,
                    "Interval": label,
                    "Price Change %": round(avg_price_change * 100, 2),
                    "Metric Change %": round(metric_change * 100, 2),
                    "Score": round(similarity, 4),
                    "Direction": "Positive" if metric_change * avg_price_change > 0 else "Negative",
                    "Company": ticker_symbol
                })

    except Exception as e:
        print(f"❌ Error processing {ticker_symbol}: {e}")
        continue

# === Save final results ===
if all_results:
    final_df = pd.DataFrame(all_results)
    combined_output_path = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/Correlations/all_companies_correlation_summary.csv"
    final_df.to_csv(combined_output_path, index=False)
    print(f"✅ Final summary saved to:\n{combined_output_path}")
else:
    print("⚠️ No valid data processed.")


🔍 Processing ACCL.TA...
🔍 Processing ACKR.TA...
🔍 Processing ARDM.TA...
🔍 Processing AFHL.TA...
🔍 Processing AFPR.TA...
🔍 Processing AFRE.TA...
🔍 Processing ARPT.TA...
🔍 Processing ARTS.TA...
🔍 Processing ALBA.TA...
🔍 Processing ALLT.TA...
🔍 Processing AMDA.TA...


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AMDA.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-12-31 -> 2025-01-01)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AMDA.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-12-30 -> 2024-12-31)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AMDA.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-12-29 -> 2024-12-30)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AMDA.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-12-28 -> 2024-12-29)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AMDA.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-12-27 -> 2024-12-28)')


⚠️ Skipping AMDA.TA — missing price data.
🔍 Processing ALMA.TA...
🔍 Processing ALUMA.TA...
🔍 Processing AMRK.TA...
🔍 Processing AMOT.TA...
🔍 Processing APLP.TA...
🔍 Processing ARD.TA...
🔍 Processing ARF.TA...
🔍 Processing ASHG.TA...
🔍 Processing ASGR.TA...
🔍 Processing AUDC.TA...
🔍 Processing AUSA-M.TA...
🔍 Processing AMX.TA...
🔍 Processing AVGL.TA...
🔍 Processing AVIA.TA...
🔍 Processing AILN.TA...
🔍 Processing AZRG.TA...
🔍 Processing BCOM.TA...
🔍 Processing BIG.TA...


ERROR:yfinance:
1 Failed download:


🔍 Processing BLITZ-M.TA...


ERROR:yfinance:['BLITZ-M.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-30 -> 2024-10-01)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BLITZ-M.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-29 -> 2024-09-30)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BLITZ-M.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-28 -> 2024-09-29)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BLITZ-M.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-27 -> 2024-09-28)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BLITZ-M.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-26 -> 2024-09-27)')


⚠️ Skipping BLITZ-M.TA — missing price data.
🔍 Processing BLSR.TA...
🔍 Processing BONS.TA...
🔍 Processing BWAY.TA...
🔍 Processing CLAB.TA...
🔍 Processing CAMT.TA...
🔍 Processing CPTP.TA...
🔍 Processing CRSM.TA...
🔍 Processing CRSR.TA...
🔍 Processing CRML.TA...
🔍 Processing CRMT.TA...
🔍 Processing CAST.TA...
🔍 Processing CEL.TA...
🔍 Processing CPIA.TA...
🔍 Processing CFX.TA...
🔍 Processing CGEN.TA...
🔍 Processing DANH.TA...
🔍 Processing DLEKG.TA...
🔍 Processing DELG.TA...
🔍 Processing DIPL.TA...
🔍 Processing DISI.TA...
🔍 Processing DRSH.TA...
🔍 Processing ELAL.TA...
🔍 Processing EMITF-M.TA...
🔍 Processing ESLT.TA...
🔍 Processing ELCO.TA...
🔍 Processing ECP.TA...
🔍 Processing ELCRE.TA...
🔍 Processing ELWS.TA...
🔍 Processing ELLO.TA...
🔍 Processing ELSPC.TA...
🔍 Processing ENOG.TA...
🔍 Processing ENLV.TA...
🔍 Processing EQTL.TA...
🔍 Processing FTAL.TA...
🔍 Processing FLYS.TA...
🔍 Processing FBRT.TA...
🔍 Processing FORTY.TA...
🔍 Processing GCT.TA...
🔍 Processing GVYM.TA...
🔍 Processing GEF

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['GRAC.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-30 -> 2024-10-01)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['GRAC.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-29 -> 2024-09-30)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['GRAC.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-28 -> 2024-09-29)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['GRAC.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-27 -> 2024-09-28)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['GRAC.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-12-31 -> 2025-01-01)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['GRAC.TA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-09-30 -> 2024-10-02)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['GRAC.T

🔍 Processing G107.TA...
🔍 Processing HGG.TA...
🔍 Processing MSBI.TA...
🔍 Processing HAMAT.TA...
🔍 Processing HICN.TA...
🔍 Processing HLAN.TA...
🔍 Processing HIPR.TA...
🔍 Processing HIVE.TA...
🔍 Processing HLMS.TA...
🔍 Processing HMGS.TA...
🔍 Processing HUMX.TA...
🔍 Processing IDNT.TA...
🔍 Processing IDMO.TA...
🔍 Processing ILX.TA...
🔍 Processing INRM.TA...
🔍 Processing INCR.TA...
🔍 Processing ISHI.TA...
🔍 Processing ISRG.TA...
🔍 Processing ISRO.TA...
🔍 Processing ISTA.TA...
🔍 Processing JNGO.TA...
🔍 Processing KAFR.TA...
🔍 Processing KSTN.TA...
🔍 Processing KLIL.TA...
🔍 Processing LAHAV.TA...
🔍 Processing LSCO.TA...
🔍 Processing LBRA.TA...
🔍 Processing LCTX.TA...
🔍 Processing LPSN.TA...
🔍 Processing MGIC.TA...
🔍 Processing MTLF.TA...
🔍 Processing MTRX.TA...
🔍 Processing MAXO.TA...
🔍 Processing MTRN.TA...
🔍 Processing MEDN.TA...
🔍 Processing MLSR.TA...
🔍 Processing MNIN.TA...
🔍 Processing MZTF.TA...
🔍 Processing MLRN.TA...
🔍 Processing NFTA.TA...
🔍 Processing NYAX.TA...
🔍 Processing NTG

In [ ]:
final_df

,Metric,Interval,Price Change %,Metric Change %,Score,Direction,Company
0,Unnamed: 0,1 Day,0.70,0.00,0.9930,Negative,ACCL.TA
1,Earnings Per Share (EPS),1 Day,0.70,0.00,0.9930,Negative,ACCL.TA
2,Number of Outstanding Shares,1 Day,0.70,0.00,0.9930,Negative,ACCL.TA
3,Unnamed: 0,1 month (30 days),22.78,0.00,0.7722,Negative,ACCL.TA
4,Earnings Per Share (EPS),1 month (30 days),22.78,0.00,0.7722,Negative,ACCL.TA
...,...,...,...,...,...,...,...
3663,Earnings Per Share (EPS),1 quarter (90 days),20.21,0.00,0.7979,Negative,ZUR.TA
3664,Gross Profit,1 quarter (90 days),20.21,158.13,-0.3793,Positive,ZUR.TA
3665,Operating Income (EBIT),1 quarter (90 days),20.21,248.44,-1.0000,Positive,ZUR.TA
3666,Total Assets,1 quarter (90 days),20.21,-1.17,0.7863,Negative,ZUR.TA


## Creating 3 different dataframes for score, price change & metric change across different intervals listed above. All dataframes will be saved to drive as csv files.

In [ ]:
# Re-attempt: Use pivot without grouping by levels that cause length mismatch
# Instead, create a new DataFrame manually with nested lists for each metric per company

# Start from scratch to avoid grouping issues
result = {}

# Populate manually by iterating over unique companies and metrics
for company in final_df['Company'].unique():
    company_data = final_df[final_df['Company'] == company]
    row = {}
    for metric in final_df['Metric'].unique():
        metric_scores = []
        for interval in ['1 Day', '1 month (30 days)', '1 quarter (90 days)']:
            match = company_data[
                (company_data['Metric'] == metric) &
                (company_data['Interval'] == interval)
            ]
            if not match.empty:
                metric_scores.append(round(match['Score'].values[0], 4))
            else:
                metric_scores.append(None)  # Or np.nan
        row[metric] = metric_scores
    result[company] = row

# Convert to final DataFrame
final_score_df = pd.DataFrame.from_dict(result, orient='index')
final_score_df = final_score_df.drop(columns=['Unnamed: 0'])
final_score_df


,Earnings Per Share (EPS),Number of Outstanding Shares,Revenue,Net Income,Gross Profit,Operating Income (EBIT),Total Assets,Long-Term Debt
ACCL.TA,"[0.993, 0.7722, 0.7504]","[0.993, 0.7722, 0.7504]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]"
ACKR.TA,"[0.9682, 0.9448, 0.8475]","[0.9682, 0.9448, 0.8475]","[-1.0, -1.0, -1.0]","[-0.1186, -0.0952, 0.0021]","[-1.0, -1.0, -1.0]","[-1.0, -1.0, -0.9123]","[0.9923, 0.9689, 0.8717]","[0.8907, 0.8673, 0.77]"
ARDM.TA,"[0.9889, 0.0663, 0.6737]","[0.9889, 0.0663, 0.6737]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]"
AFHL.TA,"[0.9918, 0.951, 0.5437]","[0.9918, 0.951, 0.5437]","[-1.0, -1.0, -1.0]","[-1.0, -1.0, -1.0]","[-1.0, -1.0, -1.0]","[-0.8973, -0.8565, -0.4492]","[0.9463, 0.9871, 0.6057]","[0.9921, 0.9671, 0.5599]"
AFPR.TA,"[0.9991, 0.9736, 0.9956]","[0.9991, 0.9736, 0.9956]","[-1.0, -1.0, -1.0]","[0.0872, 0.1127, 0.0819]","[-1.0, -1.0, -1.0]","[-1.0, -1.0, -1.0]","[0.9787, 0.9532, 0.984]","[0.9904, 0.9841, 0.9851]"
...,...,...,...,...,...,...,...,...
XTLB.TA,"[0.9957, 0.8137, 0.7184]","[0.9957, 0.8137, 0.7184]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]"
YBOX.TA,"[0.9839, 0.8796, 0.7411]","[0.9839, 0.8796, 0.7411]","[-0.8149, -0.6784, -0.5399]","[0.9909, 0.8726, 0.7341]","[-1.0, -1.0, -0.9756]","[-1.0, -1.0, -1.0]","[0.9052, 0.9583, 0.8198]","[0.86, 0.9965, 0.865]"
ZNKL.TA,"[0.9993, 0.9906, 0.9918]","[0.9993, 0.9906, 0.9918]","[-1.0, -1.0, -1.0]","[-1.0, -0.9993, -1.0]","[-1.0, -1.0, -1.0]","[-1.0, -1.0, -1.0]","[0.9632, 0.9718, 0.9707]","[-0.0007, -0.0094, -0.0082]"
ZOOZ.TA,"[0.9667, 0.9672, 0.9934]","[0.9667, 0.9672, 0.9934]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]"


In [ ]:
# Re-attempt: Use pivot without grouping by levels that cause length mismatch
# Instead, create a new DataFrame manually with nested lists for each metric per company

# Start from scratch to avoid grouping issues
result = {}

# Populate manually by iterating over unique companies and metrics
for company in final_df['Company'].unique():
    company_data = final_df[final_df['Company'] == company]
    row = {}
    for metric in final_df['Metric'].unique():
        metric_scores = []
        for interval in ['1 Day', '1 month (30 days)', '1 quarter (90 days)']:
            match = company_data[
                (company_data['Metric'] == metric) &
                (company_data['Interval'] == interval)
            ]
            if not match.empty:
                metric_scores.append(round(match['Price Change %'].values[0], 4))
            else:
                metric_scores.append(None)  # Or np.nan
        row[metric] = metric_scores
    result[company] = row

# Convert to final DataFrame
final_price_change_df = pd.DataFrame.from_dict(result, orient='index')
final_price_change_df = final_price_change_df.drop(columns=['Unnamed: 0'])
final_price_change_df


,Earnings Per Share (EPS),Number of Outstanding Shares,Revenue,Net Income,Gross Profit,Operating Income (EBIT),Total Assets,Long-Term Debt
ACCL.TA,"[0.7, 22.78, 24.96]","[0.7, 22.78, 24.96]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]"
ACKR.TA,"[3.18, 5.52, 15.25]","[3.18, 5.52, 15.25]","[3.18, 5.52, 15.25]","[3.18, 5.52, 15.25]","[3.18, 5.52, 15.25]","[3.18, 5.52, 15.25]","[3.18, 5.52, 15.25]","[3.18, 5.52, 15.25]"
ARDM.TA,"[-1.11, 93.37, 32.63]","[-1.11, 93.37, 32.63]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]"
AFHL.TA,"[0.82, 4.9, 45.63]","[0.82, 4.9, 45.63]","[0.82, 4.9, 45.63]","[0.82, 4.9, 45.63]","[0.82, 4.9, 45.63]","[0.82, 4.9, 45.63]","[0.82, 4.9, 45.63]","[0.82, 4.9, 45.63]"
AFPR.TA,"[0.09, 2.64, -0.44]","[0.09, 2.64, -0.44]","[0.09, 2.64, -0.44]","[0.09, 2.64, -0.44]","[0.09, 2.64, -0.44]","[0.09, 2.64, -0.44]","[0.09, 2.64, -0.44]","[0.09, 2.64, -0.44]"
...,...,...,...,...,...,...,...,...
XTLB.TA,"[0.43, -18.63, -28.16]","[0.43, -18.63, -28.16]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]"
YBOX.TA,"[-1.61, 12.04, 25.89]","[-1.61, 12.04, 25.89]","[-1.61, 12.04, 25.89]","[-1.61, 12.04, 25.89]","[-1.61, 12.04, 25.89]","[-1.61, 12.04, 25.89]","[-1.61, 12.04, 25.89]","[-1.61, 12.04, 25.89]"
ZNKL.TA,"[0.07, 0.94, 0.82]","[0.07, 0.94, 0.82]","[0.07, 0.94, 0.82]","[0.07, 0.94, 0.82]","[0.07, 0.94, 0.82]","[0.07, 0.94, 0.82]","[0.07, 0.94, 0.82]","[0.07, 0.94, 0.82]"
ZOOZ.TA,"[-3.33, 3.28, -0.66]","[-3.33, 3.28, -0.66]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]","[None, None, None]"


In [ ]:
# Now apply your transformation to get Metric Change % (single value per company/metric)
metric_change_result = {}

for company in final_df['Company'].unique():
    company_data = final_df[final_df['Company'] == company]
    row = {}
    for metric in final_df['Metric'].unique():
        match = company_data[company_data['Metric'] == metric]
        if not match.empty:
            row[metric] = round(match['Metric Change %'].values[0], 4)
        else:
            row[metric] = None
    metric_change_result[company] = row

# Convert to DataFrame
final_metric_change_df = pd.DataFrame.from_dict(metric_change_result, orient='index')
final_metric_change_df = final_metric_change_df.drop(columns=['Unnamed: 0'])
final_metric_change_df

,Earnings Per Share (EPS),Number of Outstanding Shares,Revenue,Net Income,Gross Profit,Operating Income (EBIT),Total Assets,Long-Term Debt
ACCL.TA,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
ACKR.TA,0.0,0.0,257.17,115.04,241.12,206.47,2.41,-7.75
ARDM.TA,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
AFHL.TA,0.0,0.0,289.48,322.78,250.99,190.55,6.19,1.61
AFPR.TA,0.0,0.0,275.73,91.37,273.35,282.44,-2.04,1.05
...,...,...,...,...,...,...,...,...
XTLB.TA,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
YBOX.TA,0.0,0.0,179.88,-0.70,223.45,-420.18,7.87,12.39
ZNKL.TA,0.0,0.0,239.15,200.87,228.01,211.66,3.76,-100.00
ZOOZ.TA,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import os
from pathlib import Path

# Your actual file paths from earlier
csv_path_1 = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/report_analysis_2024_q3.csv"
csv_path_2 = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/report_analysis_2024_q4.csv"

# Base directory for correlation outputs
base_dir = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/Correlations"
subdirs = ["correlation score", "correlations price change", "correlation metrics change"]

# Ensure all target directories exist
for sub in subdirs:
    full_path = Path(base_dir) / sub
    full_path.mkdir(parents=True, exist_ok=True)

# Generate filename suffix like q3_2024_to_q4_2024
def extract_q_year(path):
    import re
    match = re.search(r'report_analysis_(\d{4})_q([1-4])', path.lower())
    return f"Q{match.group(2)}_{match.group(1)}" if match else "Unknown"

q3_suffix = extract_q_year(csv_path_1)
q4_suffix = extract_q_year(csv_path_2)
file_suffix = f"{q3_suffix}_to_{q4_suffix}.csv"
# Convert all elements from np.float64 to native float inside lists
final_score_df = final_score_df.applymap(lambda cell: [float(x) if x is not None else None for x in cell] if isinstance(cell, list) else cell)
final_price_change_df = final_price_change_df.applymap(lambda cell: [float(x) if x is not None else None for x in cell] if isinstance(cell, list) else cell)
final_metric_change_df = final_metric_change_df.applymap(lambda cell: [float(x) if x is not None else None for x in cell] if isinstance(cell, list) else cell)

# Save the DataFrames
final_score_df.to_csv(f"{base_dir}/correlation score/{file_suffix}")
final_price_change_df.to_csv(f"{base_dir}/correlations price change/{file_suffix}")
final_metric_change_df.to_csv(f"{base_dir}/correlation metrics change/{file_suffix}")

print("✅ All correlation CSVs saved successfully.")


<ipython-input-7-ee816b547ab3>:27: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  final_score_df = final_score_df.applymap(lambda cell: [float(x) if x is not None else None for x in cell] if isinstance(cell, list) else cell)
<ipython-input-7-ee816b547ab3>:28: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  final_price_change_df = final_price_change_df.applymap(lambda cell: [float(x) if x is not None else None for x in cell] if isinstance(cell, list) else cell)
<ipython-input-7-ee816b547ab3>:29: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  final_metric_change_df = final_metric_change_df.applymap(lambda cell: [float(x) if x is not None else None for x in cell] if isinstance(cell, list) else cell)


✅ All correlation CSVs saved successfully.
